# Data Ingestion Pipeline

In [16]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [17]:
def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # IMPORTANT: Capture the 'Subject' from the folder name
            subject_name = pdf_file.parent.name 

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['subject'] = subject_name # This is key for UPSC routing
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
        except Exception as e:
            print(f"Error loading {pdf_file.name}: {e}")
            
    print(f"\nTotal pages loaded: {len(all_documents)}")
    return all_documents

In [18]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    
    if split_docs:
        print(f"Split {len(documents)} pages into {len(split_docs)} chunks")
        print(f"\nExample chunk Metadata: {split_docs[0].metadata}")
        
    return split_docs

In [19]:
input_path = "documents" 

all_pdf_documents = process_all_pdfs(input_path)
chunks = split_documents(all_pdf_documents)

Found 142 PDF files to process
Processing jess201.pdf...
Processing jess202.pdf...
Processing jess203.pdf...
Processing jess204.pdf...
Processing jess205.pdf...
Processing jess2ps.pdf...
Processing keec101.pdf...
Processing keec102.pdf...
Processing keec103.pdf...
Processing keec104.pdf...
Processing keec105.pdf...
Processing keec106.pdf...
Processing keec107.pdf...
Processing keec108.pdf...
Processing keec1gl.pdf...
Processing keec1ps.pdf...
Processing leec101.pdf...
Processing leec102.pdf...
Processing leec103.pdf...
Processing leec104.pdf...
Processing leec105.pdf...
Processing leec106.pdf...
Processing leec1ps.pdf...
Processing leec201.pdf...
Processing leec202.pdf...
Processing leec203.pdf...
Processing leec204.pdf...
Processing leec205.pdf...
Processing leec2gl.pdf...
Processing leec2ps.pdf...
Processing iess101.pdf...
Processing iess102.pdf...
Processing iess103.pdf...
Processing iess104.pdf...
Processing iess105.pdf...
Processing iess106.pdf...
Processing iess1ps.pdf...
Process

KeyboardInterrupt: 

In [ ]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in documents:", os.listdir("documents"))

Current Working Directory: c:\Users\M.Hema Siri Ramya\OneDrive\Desktop\RAG\upscRAG\myvenv
Files in documents: ['economics', 'geography', 'history', 'politics']


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
# 1. Initialize Local Embeddings (Downloads once ~300MB, then runs offline)
model_name = "BAAI/bge-small-en-v1.5"
encode_kwargs = {'normalize_embeddings': True} # Better for similarity search
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    encode_kwargs=encode_kwargs
)

c:\Users\M.Hema Siri Ramya\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\M.Hema Siri Ramya\.cache\huggingface\hub\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8174.75it/s

In [ ]:
def build_subject_experts(chunks):
    # Group your chunks by subject
    subject_map = {}
    for chunk in chunks:
        subj = chunk.metadata.get('subject', 'general')
        if subj not in subject_map:
            subject_map[subj] = []
        subject_map[subj].append(chunk)

    # Create a unique database for each subject
    for subject, subject_chunks in subject_map.items():
        print(f"📦 Storing {len(subject_chunks)} chunks in {subject.upper()} Expert DB...")
        
        # This creates a folder like ./vector_dbs/history
        persist_dir = f"./vector_dbs/{subject}"
        
        Chroma.from_documents(
            documents=subject_chunks,
            embedding=embeddings,
            persist_directory=persist_dir
        )
    print("\n✅ All Subject-Specific Databases are ready!")

In [ ]:
subject_experts = build_subject_experts(chunks)

📦 Storing 1579 chunks in ECONOMICS Expert DB...
📦 Storing 1184 chunks in GEOGRAPHY Expert DB...
📦 Storing 1486 chunks in HISTORY Expert DB...
📦 Storing 2091 chunks in POLITICS Expert DB...

✅ All Subject-Specific Databases are ready!


## Agentic RAG 

In [41]:
## Agentic RAG 
from langchain.tools import tool
from langchain_chroma import Chroma

# Re-use the same embedding model you used for building
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

@tool
def search_history(query: str):
    """Search for facts about Indian Constitution. Input is a search string."""
    db = Chroma(persist_directory="./vector_dbs/history", embedding_function=embeddings)
    
    # Use a similarity score threshold to detect "no match"
    docs_with_scores = db.similarity_search_with_relevance_scores(query, k=3)
    
    # If the best match is poor (e.g., score < 0.5), tell the agent to look elsewhere
    if not docs_with_scores or docs_with_scores[0][1] < 0.5:
        return "No relevant information found in the History textbooks. Please try a web search."
    
    return "\n\n".join([d[0].page_content for d in docs_with_scores])

@tool
def search_polity(query: str):
    """Search for facts about Indian Constitution. Input is a search string."""
    db = Chroma(persist_directory="./vector_dbs/politics", embedding_function=embeddings)
    
    # Use a similarity score threshold to detect "no match"
    docs_with_scores = db.similarity_search_with_relevance_scores(query, k=3)
    
    # If the best match is poor (e.g., score < 0.5), tell the agent to look elsewhere
    if not docs_with_scores or docs_with_scores[0][1] < 0.5:
        return "No relevant information found in the Polity textbooks. Please try a web search."
    
    return "\n\n".join([d[0].page_content for d in docs_with_scores])

@tool
def search_geography(query: str):
    """Search for facts about Indian Constitution. Input is a search string."""
    db = Chroma(persist_directory="./vector_dbs/geography", embedding_function=embeddings)
    
    # Use a similarity score threshold to detect "no match"
    docs_with_scores = db.similarity_search_with_relevance_scores(query, k=3)
    
    # If the best match is poor (e.g., score < 0.5), tell the agent to look elsewhere
    if not docs_with_scores or docs_with_scores[0][1] < 0.5:
        return "No relevant information found in the Geography textbooks. Please try a web search."
    
    return "\n\n".join([d[0].page_content for d in docs_with_scores])

@tool
def search_economics(query: str):
    """Search for facts about Indian Constitution. Input is a search string."""
    db = Chroma(persist_directory="./vector_dbs/economics", embedding_function=embeddings)
    
    # Use a similarity score threshold to detect "no match"
    docs_with_scores = db.similarity_search_with_relevance_scores(query, k=3)
    
    # If the best match is poor (e.g., score < 0.5), tell the agent to look elsewhere
    if not docs_with_scores or docs_with_scores[0][1] < 0.5:
        return "No relevant information found in the Economics textbooks. Please try a web search."
    
    return "\n\n".join([d[0].page_content for d in docs_with_scores])

# Add similar tools for Geography, Economy, etc.
tools = [search_history, search_polity,search_geography,search_economics]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8404.49it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [52]:
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from dotenv import load_dotenv
import os

load_dotenv()

groq_api = os.getenv("GROQ_API")

llm = ChatGroq(
    groq_api_key=groq_api,
    model="llama-3.1-8b-instant",
    temperature=0,
)
system_prompt = """You are a specialized UPSC CSE Assistant. 
Your goal is to provide accurate, multi-dimensional answers (Introduction, Body, Conclusion).
Always cite the source material from your databases."""

agent = create_react_agent(llm, tools, prompt=system_prompt)

query = "Socialism in Europe and the Russian Revolution"

result = agent.invoke({
    "messages": [
        ("human", query)
    ]
})

print(result["messages"][-1].content)

The Russian Revolution had a significant impact on the development of socialism in Europe. The revolution led to the establishment of the Soviet Union, which became a model for socialist development in other countries. The Soviet Union's socialist system, which was based on the principles of collective ownership of the means of production, central planning, and the distribution of goods and services based on need rather than market principles, was seen as a viable alternative to capitalism.

In Europe, the Russian Revolution inspired the development of socialist movements and parties, which sought to establish socialist systems in their own countries. The socialist movement in Europe was influenced by the ideas of Karl Marx and Friedrich Engels, who argued that socialism was a necessary step towards the establishment of a communist society.

The Russian Revolution also had a significant impact on the development of socialism in India. The Indian National Congress, which was the main na

In [54]:
test_questions = [
    # 1. History - Events & Processes
    {
        "topic": "Socialism in Europe & Russian Revolution",
        "question": "What was the significance of the 'April Theses' proposed by Vladimir Lenin in 1917?",
        "expected_answer": "Demanded an end to the war, transfer of land to peasants, and nationalization of banks."
    },
    # 2. History - Nazism
    {
        "topic": "Nazism and the Rise of Hitler",
        "question": "Explain the significance of the Enabling Act passed in March 1933.",
        "expected_answer": "Established a legal dictatorship, allowing Hitler to sideline Parliament and rule by decree."
    },
    # 3. Geography - Physical
    {
        "topic": "Physical Features of India",
        "question": "Differentiate between 'Bhadar' and 'Terai' regions in the Northern Plains.",
        "expected_answer": "Bhabar is a pebble-studded zone where streams disappear; Terai is a marshy, wet, and forested region where streams re-emerge."
    },
    # 4. Geography - Drainage/Climate
    {
        "topic": "Drainage Systems",
        "question": "Why do the Himalayan rivers have a perennial nature compared to Peninsular rivers?",
        "expected_answer": "They receive water from both rainfall and the melting of Himalayan glaciers, unlike Peninsular rivers which depend solely on rain."
    },
    # 5. Resources & Economy
    {
        "topic": "Minerals and Energy Resources",
        "question": "Identify the 'Big Three' coal-producing states in India mentioned in the NCERT.",
        "expected_answer": "Jharkhand, Odisha, and Chhattisgarh."
    }
]

In [55]:
import time

def run_mini_benchmark(agent, test_set):
    summary_results = []
    for item in test_set:
        print(f"--- Evaluating: {item['topic']} ---")
        try:
            # 1. Ask the Agent (passing only the current message to keep history clean)
            res = agent.invoke({"messages": [("human", item['question'])]})
            agent_output = res["messages"][-1].content
            
            # 2. Log result
            summary_results.append({"Topic": item['topic'], "Status": "Success"})
            
            # 3. COOLDOWN: Wait 15 seconds to let the TPM 'bucket' refill
            print("Cooling down to prevent rate limits...")
            time.sleep(15) 
            
        except Exception as e:
            print(f"Error on {item['topic']}: {e}")
            summary_results.append({"Topic": item['topic'], "Status": "Failed"})
            
    return summary_results

In [56]:
print(run_mini_benchmark(agent,test_questions))

--- Evaluating: Socialism in Europe & Russian Revolution ---
Cooling down to prevent rate limits...
--- Evaluating: Nazism and the Rise of Hitler ---
Cooling down to prevent rate limits...
--- Evaluating: Physical Features of India ---
Cooling down to prevent rate limits...
--- Evaluating: Drainage Systems ---
Cooling down to prevent rate limits...
--- Evaluating: Minerals and Energy Resources ---
Cooling down to prevent rate limits...
[{'Topic': 'Socialism in Europe & Russian Revolution', 'Status': 'Success'}, {'Topic': 'Nazism and the Rise of Hitler', 'Status': 'Success'}, {'Topic': 'Physical Features of India', 'Status': 'Success'}, {'Topic': 'Drainage Systems', 'Status': 'Success'}, {'Topic': 'Minerals and Energy Resources', 'Status': 'Success'}]
